In [1]:
from google.colab import drive; drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/c2_htr'; RAG = f'{ROOT}/rag'
!pip -q install rapidfuzz sentence-transformers openpyxl
import os, re, glob, json, random, unicodedata, collections
import numpy as np, pandas as pd
os.makedirs(RAG, exist_ok=True)

SIN = re.compile(r'[\u0D80-\u0DFF]')
PHONE = re.compile(r'(\+94|0)\s?\d{2}\s?\d{3}\s?\d{4}')
NIC = re.compile(r'\b(\d{9}[VvXx]|\d{12})\b')
PUNCT = '"\'.,;:!?()[]{}<>|/\\-–—_*#~`“”‘’'
norm = lambda t: unicodedata.normalize('NFC', str(t))
mask = lambda t: NIC.sub('<NIC>', PHONE.sub('<PHONE>', t))
tokens = lambda t: [w.strip(PUNCT) for w in norm(t).split() if w.strip(PUNCT)]
def clean(t):
    t = norm(t).replace('_x000D_', ' ')          # Excel line-break junk
    return ' '.join(mask(t).split())[:500]       # one line, max 500 characters

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.0 MB/s eta 0:00:00


In [2]:
docs = []
# Municipal Council files: one column, no header row
for name in ['mc_engineering', 'mc_health']:
    d = pd.read_excel(f'{ROOT}/corpus/{name}.xlsx', header=None, dtype=str).fillna('')
    for i, t in enumerate(d[0]):
        t = clean(t)
        if len(t) > 5: docs.append({'id': f'{name}_{i}', 'text': t, 'source': name, 'category': ''})

# LECO file: has a header row (Category, Description), English text
d = pd.read_excel(f'{ROOT}/corpus/leco_final.xlsx', dtype=str).fillna('')
for i, r in d.iterrows():
    t = clean(r['Description'])
    if len(t) > 5: docs.append({'id': f'leco_{i}', 'text': t, 'source': 'leco', 'category': r['Category']})

# your own TRAIN labels only (never val/test -> no cheating in evaluation)
sp = pd.read_csv(f'{ROOT}/splits.csv', dtype=str, keep_default_na=False)
for r in sp[sp.split == 'train'].itertuples():
    docs.append({'id': r.file, 'text': clean(r.text), 'source': 'own_train_labels', 'category': ''})

# remove exact duplicates
seen, uniq = set(), []
for doc in docs:
    key = doc['text'].lower()
    if key not in seen: seen.add(key); uniq.append(doc)
docs = uniq
texts = [d['text'] for d in docs]
print(len(docs), 'docs after de-duplication |', collections.Counter(d['source'] for d in docs))
print('Sinhala docs:', sum(bool(SIN.search(t)) for t in texts))

7382 docs after de-duplication | Counter({'leco': 6647, 'mc_engineering': 393, 'own_train_labels': 215, 'mc_health': 127})
Sinhala docs: 661


In [3]:
cnt = collections.Counter(w for t in texts for w in tokens(t) if SIN.search(w) or w.isalpha())
lexicon = sorted(w for w, c in cnt.items() if c >= 2 and len(w) > 1)
places = ['කඩුවෙල', 'බත්තරමුල්ල', 'මාලඹේ', 'තලංගම', 'අතුරුගිරිය', 'මහරගම', 'නුගේගොඩ', 'මිරිහාන',
          'Kaduwela', 'Battaramulla', 'Malabe', 'Thalangama', 'Athurugiriya', 'Maharagama', 'Nugegoda', 'LECO']
lexicon = sorted(set(lexicon) | set(places))
open(f'{RAG}/lexicon.txt', 'w', encoding='utf-8').write('\n'.join(lexicon))
sin_lex = [w for w in lexicon if SIN.search(w)]
print(len(lexicon), 'lexicon words |', len(sin_lex), 'Sinhala')
print('top Sinhala words:', [w for w, c in cnt.most_common(400) if SIN.search(w)][:20])

3378 lexicon words | 583 Sinhala
top Sinhala words: ['සම්බන්ධයෙන්', 'කිරීම', 'කර', 'ඇති', 'ගැනීම', 'ජලය', 'මාර්ගය', 'අනවසර', 'සම්බන්ධව', 'වැසි', 'අවහිර', 'වන', 'රැගෙන', 'හා', 'පාර', 'නිසා']


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sentence_transformers import SentenceTransformer

tfv = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), sublinear_tf=True)
TF = tfv.fit_transform(texts)
enc = SentenceTransformer('intfloat/multilingual-e5-base')
DE = enc.encode(['passage: ' + t for t in texts], normalize_embeddings=True, batch_size=32, show_progress_bar=True)

def search_tfidf(q, k, exclude=()):
    s = linear_kernel(tfv.transform([q]), TF).ravel()
    return [i for i in s.argsort()[::-1][:k + len(exclude)] if i not in exclude][:k]
def search_dense(q, k, exclude=()):
    s = DE @ enc.encode(['query: ' + q], normalize_embeddings=True)[0]
    return [i for i in s.argsort()[::-1][:k + len(exclude)] if i not in exclude][:k]
def search_hybrid(q, k, exclude=(), pool=20):
    sc = collections.defaultdict(float)
    for ranking in (search_tfidf(q, pool, exclude), search_dense(q, pool, exclude)):
        for r, i in enumerate(ranking): sc[i] += 1 / (60 + r)      # reciprocal rank fusion
    return sorted(sc, key=sc.get, reverse=True)[:k]
METHODS = {'tfidf-char': search_tfidf, 'dense-e5': search_dense, 'hybrid': search_hybrid}

# quick sanity check with one noisy OCR line from your results
q = 'ාලඹේ පසෙරන්සිප් පැර්ක්'
for m, fn in METHODS.items(): print(m, '->', [texts[i][:60] for i in fn(q, 3)])

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/231 [00:00<?, ?it/s]

tfidf-char -> ['මාලඹේ', 'මාලඹේ | අක්\u200dරිය වීම', 'නව නුවර පාර, මාලඹේ']
dense-e5 -> ['wronk tarriff', 'power down in Moratumulla area', 'Hunupitiya railway station Around']
hybrid -> ['මාලඹේ', 'wronk tarriff', 'මාලඹේ | අක්\u200dරිය වීම']


In [6]:
rows = []
def ctx_for(g, fn, k, use_lex):
    ctx = set(lex_set) if use_lex else set()
    if fn:
        leak = {i for i in sin_idx if any(fuzz.ratio(texts[i], tl) >= 90 for tl in g.truth)}
        ctx |= set(w for i in fn(' '.join(g.pred), k, leak) for w in tokens(texts[i]))
    return ctx

configs = [('lexicon only', None, 0, True)]
for m in ['tfidf-char', 'dense-e5', 'hybrid']:
    for k in (3, 5, 10, 20):
        configs += [(m + ' only', METHODS[m], k, False), (m + ' + lexicon', METHODS[m], k, True)]

for name, fn, k, use_lex in configs:
    cov, fix = [], []
    for page, g in val_pages:
        truth_words = [w for t in g.truth for w in tokens(t) if SIN.search(w)]
        ocr_words = set(w for p in g.pred for w in tokens(p))
        wrong = [w for w in truth_words if w not in ocr_words]
        ctx = ctx_for(g, fn, k, use_lex)
        if truth_words: cov.append(np.mean([w in ctx for w in truth_words]))
        if wrong: fix.append(np.mean([w in ctx for w in wrong]))
    rows.append((name, k, len(ctx), np.mean(cov), np.mean(fix) if fix else 0))
res = pd.DataFrame(rows, columns=['config', 'top_k', 'context_words(last page)', 'coverage', 'fixable'])
print(res.round(3).to_string(index=False)); res.to_csv(f'{RAG}/retrieval_eval_val.csv', index=False)

              config  top_k  context_words(last page)  coverage  fixable
        lexicon only      0                      3378     0.637    0.554
     tfidf-char only      3                        12     0.193    0.078
tfidf-char + lexicon      3                      3379     0.646    0.554
     tfidf-char only      5                        37     0.229    0.115
tfidf-char + lexicon      5                      3384     0.646    0.554
     tfidf-char only     10                        52     0.327    0.221
tfidf-char + lexicon     10                      3385     0.646    0.554
     tfidf-char only     20                        88     0.424    0.297
tfidf-char + lexicon     20                      3394     0.646    0.554
       dense-e5 only      3                        37     0.055    0.015
  dense-e5 + lexicon      3                      3382     0.637    0.554
       dense-e5 only      5                        50     0.055    0.015
  dense-e5 + lexicon      5                      33

In [10]:
import joblib
with open(f'{RAG}/past_complaints.jsonl', 'w', encoding='utf-8') as f:
    for d in docs: f.write(json.dumps(d, ensure_ascii=False) + '\n')
joblib.dump((tfv, TF), f'{RAG}/tfidf.joblib'); np.save(f'{RAG}/dense_e5.npy', DE)
sin_lex = [w for w in lexicon if SIN.search(w)]
open(f'{RAG}/lexicon_si.txt', 'w', encoding='utf-8').write('\n'.join(sin_lex))
json.dump({'method': 'tfidf-char', 'k': 20, 'lexicon': 'lexicon_si.txt',
           'finding': 'char TF-IDF top-20 fixable 0.297 vs dense-e5 0.036 vs hybrid 0.236 on val; lexicon alone 0.554'},
          open(f'{RAG}/config.json', 'w'))
print('saved to', RAG, '|', os.listdir(RAG), '| Sinhala lexicon:', len(sin_lex))

saved to /content/drive/MyDrive/c2_htr/rag | ['lexicon.txt', 'retrieval_eval_val.csv', 'past_complaints.jsonl', 'tfidf.joblib', 'dense_e5.npy', 'lexicon_si.txt', 'config.json'] | Sinhala lexicon: 583
